# C15_E01 — Detector de drift por CUSUM

**Capítulo:** 15 — Gemelos digitales, modelos híbridos y simulación conectada  
**Identificador:** `C15_E01`  
**Objetivo pedagógico:** Detectar deriva modelo-realidad en un gemelo digital.  
**Librerías:** numpy, matplotlib

## Ejemplo industrial

Gemelo digital de horno: detección automática de drift por incrustaciones.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. CUSUM como detector de drift

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
np.random.seed(0)

def cusum(residuals, target=0.0, k=0.5, h=5.0):
    """CUSUM dos colas. Devuelve series Sp, Sn y lista de instantes de alarma."""
    Sp = 0.0; Sn = 0.0; alarms = []
    Sps, Sns = [], []
    for i, r in enumerate(residuals):
        Sp = max(0.0, Sp + (r - target) - k)
        Sn = min(0.0, Sn + (r - target) + k)
        Sps.append(Sp); Sns.append(Sn)
        if Sp > h or Sn < -h:
            alarms.append(i); Sp = 0.0; Sn = 0.0
    return np.array(Sps), np.array(Sns), alarms

# Residuos sintéticos: media 0 hasta k=200, luego sesgo lento por drift
N = 500
residuals = 0.2*np.random.randn(N)
residuals[200:] += np.linspace(0, 1.0, N-200)   # drift lineal a partir de k=200
Sp, Sn, alarms = cusum(residuals, target=0.0, k=0.3, h=4.0)
print("Alarmas en k:", alarms[:5], "...")

Alarmas en k: [340, 365, 384, 398, 410] ...


## 2. Visualización

In [2]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
ax1.plot(residuals, label="Residuos modelo–realidad")
ax1.axvline(200, color='red', alpha=0.3, label="Inicio del drift")
ax1.set_ylabel("residuo"); ax1.grid(alpha=0.3); ax1.legend()
ax1.set_title("C15_E01 — Detector de drift por CUSUM")
ax2.plot(Sp, label="$S^+$"); ax2.plot(Sn, label="$S^-$")
ax2.axhline(4.0, ls='--', color='red', alpha=0.5, label="umbral h")
ax2.axhline(-4.0, ls='--', color='red', alpha=0.5)
for a in alarms[:10]:
    ax2.axvline(a, color='orange', alpha=0.5)
ax2.set_xlabel("k"); ax2.set_ylabel("CUSUM"); ax2.legend(); ax2.grid(alpha=0.3)
out = '../../figures/cap15/fig_C15_F02_cusum.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120); plt.show()

## 3. Interpretación

CUSUM acumula desviaciones del residuo respecto al objetivo. Cuando la suma cruza el umbral `h`, declara alarma de drift y reinicia. Sensibilidad y robustez se controlan con `k` (slack) y `h` (umbral). Es un detector simple, robusto, ampliamente usado en control de calidad y en monitorización de gemelos digitales. **Tras la alarma**: recalibrar el modelo con datos recientes o reportar fallo a operador.